# 11 · Generated answers: 2 LLMs × 2 prompts over the X3 contexts (vLLM only)

Answers every question of the experiment from the retrieved X3 contexts that notebook 10 saved (`results/<experiment>/x3/contexts.jsonl`), with **two LLMs × two prompts**, and ends with one file that has, for every (question, LLM, prompt): the question, the generated answer, the known gold section(s), the retrieved parts that were given to the model, the LLM name and the prompt name. Design: `x3_to_llm.md`.

| | |
|---|---|
| LLM `qwen` | Qwen3.8-27B INT4 (`RedHatAI/Qwen3.8-27B-INT4`), the vLLM container's default |
| LLM `gemma` | Gemma 4 26B-A4B, QAT 4-bit (`cyankiwi/gemma-4-26B-A4B-it-qat-AWQ-INT4`), same container with `vllm/serve.gemma.yaml` |
| prompt `simple` | question + retrieved documents + "use only this information" |
| prompt `robust` | the same job with a system prompt (grounding, what to say when the documents do not answer, exact numbers, CS against AMC, engine type, citations) |

The prompts are in **`src/csrag/rag_prompts.py`**: edit them there. Every run re-reads that file, so no kernel restart is needed; each record stores the hash of the prompt it was made with, and a file never mixes two versions of one prompt.

> **⚠ ONLY vLLM IS NEEDED, ONE MODEL AT A TIME.** Stop the GPU encoder first (it and vLLM do not fit on the card together). Qdrant is not used.
> ```bash
> # Qwen first (the defaults of .env: serve.retrieval.yaml, RedHatAI/Qwen3.8-27B-INT4)
> docker compose stop vllm encoder-gpu && docker compose up -d vllm
> docker compose logs -f vllm                       # ~2.5 min to healthy
> ```
> The switch to Gemma is in the middle of this notebook (cell "Switch to Gemma"), and its commands are there too.

| step | what |
|---|---|
| 1 | try the first question with both prompts on the model that is up (prints the prompts and the answers, writes nothing) |
| 2 | Qwen × simple, Qwen × robust |
| 3 | **switch vLLM to Gemma in a terminal**, then check it is up |
| 4 | Gemma × simple, Gemma × robust |
| 5 | combine the four files into `rag_output.jsonl`, checks, summary |

Files in `results/<experiment>/llm/`: `rag_<llm>__<prompt>.jsonl` (one per combination: one record per question, written as the answers come in), **`rag_output.jsonl`** (the four combined; one record per question, LLM and prompt) and `summary.csv`. Every run cell **resumes**: records already in its file are kept, failed ones are redone, so a stopped cell can simply be run again. Each run cell first checks that vLLM serves *its* model and stops with the command to start it otherwise.

In [1]:
from pathlib import Path
import os, sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)  # all paths below are relative to the repo root
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")
DATA = ROOT / "data"
from csrag import experiment
EXP = experiment.active()  # `active` in experiments.toml, or the CSRAG_EXPERIMENT environment variable
RESULTS = ROOT / EXP.dir   # results/<experiment name>/: every result of this experiment
print(EXP.summary())

experiment 'run2' -> results/run2/  (Question set 1 + question set 2 (paraphrase, section_id, twin_topic, multi_section))
   100  data/eval/questions.jsonl
   177  data/eval/set2/paraphrase.jsonl
    50  data/eval/set2/section_id.jsonl
    60  data/eval/set2/twin_topic.jsonl
    90  data/eval/set2/multi_section.jsonl
   477  questions, 469 answerable | qrels: results/run2/qrels.jsonl


In [19]:
from IPython.display import display
from tqdm.auto import tqdm
from csrag import rag_generate as rg, rag_prompts

CONTEXTS = RESULTS / "x3" / "contexts.jsonl"  # written by notebook 10
OUT = RESULTS / "llm"  # every file of this notebook
PARAMS = {"temperature": 0.0, "max_tokens": 4000, "seed": 0}  # thinking is always off
LLM_ORDER = ["qwen", "gemma"]  # the order of the run cells below and of the combined file
PROMPT_ORDER = ["simple", "robust"]
LIMIT = None  # an integer runs only that many first questions per combination (a smoke test: delete the files in OUT before the full run)
TRY_INDEX = 6  # the question of the "try one question" cells

assert set(LLM_ORDER) <= set(rg.LLMS) and set(PROMPT_ORDER) <= set(rag_prompts.PROMPTS)
assert CONTEXTS.is_file(), f"{CONTEXTS} not found: run notebook 10 for experiment {EXP.name!r} first"
items = rg.load_inputs(EXP, CONTEXTS)  # the questions joined with their X3 contexts, in question order
used = items[:LIMIT] if LIMIT else items  # the questions the run cells answer
OUT.mkdir(parents=True, exist_ok=True)

print(f"{len(items)} questions ({sum(i['question']['answerable'] for i in items)} answerable); this notebook answers {len(used)} of them")
print("prompts:", {n: f"{rag_prompts.prompt_hash(n)} ({rag_prompts.PROMPTS[n]['description']})" for n in PROMPT_ORDER})
print("models:", {k: rg.LLMS[k]["model_id"] for k in LLM_ORDER})
print("vLLM serves now:", rg.served_models() or "nothing (start Qwen: docker compose stop vllm encoder-gpu && docker compose up -d vllm)")


def peek(result, n=2):
    """The counts of a run, and its first answers."""
    print(result)
    for r in rg.read_jsonl(result["file"])[:n]:
        print("-" * 100)
        print(f"{r['question_id']} | {r['llm']} x {r['prompt']} | {r['completion_tokens']} tokens, {r['seconds']} s, finish={r['finish_reason']}")
        print("Q:", r["question"])
        print("A:", r["answer"] or r["error"])


def show_try(t):
    """Print the result of rg.try_question: the question, the gold, and each prompt with its answer."""
    print(f"{t['llm']} | {t['question_id']}\nQ: {t['question']}\ngold sections: {t['gold']}\nreference answer: {t['reference_answer']}")
    for name, r in t["prompts"].items():
        print("\n" + "=" * 100 + f"\nPROMPT `{name}` as the model sees it\n" + "=" * 100)
        print(r["text"])
        print("-" * 100 + f"\nANSWER with `{name}` ({r['prompt_tokens']} prompt + {r['completion_tokens']} completion tokens, {r['seconds']} s, finish={r['finish_reason']})\n" + "-" * 100)
        print(r["answer"])

477 questions (469 answerable); this notebook answers 477 of them
prompts: {'simple': 'a5295d99 (plain RAG prompt: question, retrieved documents, use only these)', 'robust': '7a58a305 (system prompt with grounding, abstention, exactness, CS/AMC, engine type and citation rules)'}
models: {'qwen': 'RedHatAI/Qwen3.8-27B-INT4', 'gemma': 'cyankiwi/gemma-4-26B-A4B-it-qat-AWQ-INT4'}
vLLM serves now: ['RedHatAI/Qwen3.8-27B-INT4']


## Try one question

Asks whichever model vLLM serves now (Qwen or Gemma) the question at `TRY_INDEX` with **both** prompts, and prints each prompt exactly as the model sees it, the answer, the gold section(s) and the reference answer. It writes nothing; use it to see what the prompts do before the long runs, and again after editing `src/csrag/rag_prompts.py`.

In [5]:
show_try(rg.try_question(items, TRY_INDEX, PARAMS))

qwen3.8-27b-int4 | q007
Q: Which OEI power ratings may be established for turbine engines intended for multi-engined rotorcraft?
gold sections: ['CS-E 40']
reference answer: 30-Second OEI Power, 2-Minute OEI Power, 2 1/2-Minute OEI Power, 30-Minute OEI Power and Continuous OEI Power (see AMC E 40(b)(3)).

PROMPT `simple` as the model sees it
--- user ---
Here is a user question and the retrieved documents. Use only the information in the documents to answer the question.

Documents:

[1] CS-E 40 (page 30-31)
CS-E 40 Ratings (part 1/2)

(See AMC E 40)

(a) Power ratings must be established for Take-off Power and/or Thrust and for Maximum Continuous Power and/or Thrust, for all Engines.

(b) Other ratings may also be established as -

  (1) Piston Engines -

    (i) Maximum Recommended Cruising Power,

    (ii) Maximum Best Economy Cruising Power;

  (2) Turbine Engines for multi-engined aeroplanes:

    (i) 2 1/2-Minute OEI Power or Thrust,

    (ii) Continuous OEI Power or Thrust;

  (

## Qwen × simple, Qwen × robust

vLLM must serve Qwen (`RedHatAI/Qwen3.8-27B-INT4`); each cell checks it. Four requests run at once. The answers are written to their file as they arrive, and the cell resumes if it is run again.

In [20]:
peek(rg.run_combination(used, "qwen", "simple", rg.combo_path(OUT, "qwen", "simple"), PARAMS, progress=tqdm))

qwen3.8-27b-int4 x simple:   0%|          | 0/1 [00:00<?, ?it/s]

{'file': '/home/goksu/projects/ikincitik/results/run2/llm/rag_qwen3.8-27b-int4__simple.jsonl', 'records': 477, 'new': 1, 'skipped': 476, 'errors': 0, 'truncated': 0, 'seconds': 121}
----------------------------------------------------------------------------------------------------
q001 | qwen3.8-27b-int4 x simple | 33 tokens, 9.993 s, finish=stop
Q: Besides the acceptable means of compliance in Book 2 of the engine certification specifications, which other document may also provide acceptable means of compliance to the Book 1 specifications?
A: Based on document [1], **AMC-20** may also provide acceptable means of compliance to the specifications in Book 1 of the CS-E.
----------------------------------------------------------------------------------------------------
q002 | qwen3.8-27b-int4 x simple | 33 tokens, 12.322 s, finish=stop
Q: Which subparts of the engine certification specifications apply to turbine engines?
A: Based on document [1], specifically section (d), the specifica

In [17]:
peek(rg.run_combination(used, "qwen", "robust", rg.combo_path(OUT, "qwen", "robust"), PARAMS, progress=tqdm))

qwen3.8-27b-int4 x robust:   0%|          | 0/3 [00:00<?, ?it/s]

{'file': '/home/goksu/projects/ikincitik/results/run2/llm/rag_qwen3.8-27b-int4__robust.jsonl', 'records': 477, 'new': 3, 'skipped': 474, 'errors': 0, 'truncated': 0, 'seconds': 97}
----------------------------------------------------------------------------------------------------
q002 | qwen3.8-27b-int4 x robust | 21 tokens, 11.19 s, finish=stop
Q: Which subparts of the engine certification specifications apply to turbine engines?
A: The specifications of subparts A, D, E and F apply to Turbine Engines [1].
----------------------------------------------------------------------------------------------------
q001 | qwen3.8-27b-int4 x robust | 26 tokens, 14.294 s, finish=stop
Q: Besides the acceptable means of compliance in Book 2 of the engine certification specifications, which other document may also provide acceptable means of compliance to the Book 1 specifications?
A: AMC-20 may also provide acceptable means of compliance to the specifications in Book 1 of CS-E [1].


## Switch to Gemma (in a terminal)

Nothing runs in this cell. Stop Qwen and start Gemma 4 26B-A4B (**only one vLLM fits on the card**):

```bash
docker compose stop vllm
VLLM_MODEL_ID=cyankiwi/gemma-4-26B-A4B-it-qat-AWQ-INT4 VLLM_CONFIG=serve.gemma.yaml docker compose up -d vllm
docker compose logs -f vllm                       # until "Application startup complete"
```

`compose.yaml` takes the model from `VLLM_MODEL_ID` (Qwen when it is unset), `vllm/serve.gemma.yaml` has the serve options (language-model-only, 32768 tokens, 4 sequences, 0.80 of the card, like Qwen's). To come back to Qwen later: `docker compose stop vllm && docker compose up -d vllm`.

In [9]:
model = rg.wait_for_model("gemma", timeout_s=900, strict=False)  # waits through the switch, up to 15 minutes
print("vLLM serves:", rg.served_models())
show_try(rg.try_question(items, TRY_INDEX, PARAMS))  # one request per prompt, so a broken server shows before the long runs

vLLM serves: ['cyankiwi/gemma-4-26B-A4B-it-qat-AWQ-INT4']
gemma-4-26b-a4b-qat-int4 | q007
Q: Which OEI power ratings may be established for turbine engines intended for multi-engined rotorcraft?
gold sections: ['CS-E 40']
reference answer: 30-Second OEI Power, 2-Minute OEI Power, 2 1/2-Minute OEI Power, 30-Minute OEI Power and Continuous OEI Power (see AMC E 40(b)(3)).

PROMPT `simple` as the model sees it
--- user ---
Here is a user question and the retrieved documents. Use only the information in the documents to answer the question.

Documents:

[1] CS-E 40 (page 30-31)
CS-E 40 Ratings (part 1/2)

(See AMC E 40)

(a) Power ratings must be established for Take-off Power and/or Thrust and for Maximum Continuous Power and/or Thrust, for all Engines.

(b) Other ratings may also be established as -

  (1) Piston Engines -

    (i) Maximum Recommended Cruising Power,

    (ii) Maximum Best Economy Cruising Power;

  (2) Turbine Engines for multi-engined aeroplanes:

    (i) 2 1/2-Minute O

## Gemma × simple, Gemma × robust

vLLM must serve Gemma (`cyankiwi/gemma-4-26B-A4B-it-qat-AWQ-INT4`); each cell checks it. Same parameters, same four-at-once, same resuming as for Qwen.

In [14]:
peek(rg.run_combination(used, "gemma", "simple", rg.combo_path(OUT, "gemma", "simple"), PARAMS, progress=tqdm))

gemma-4-26b-a4b-qat-int4 x simple:   0%|          | 0/2 [00:00<?, ?it/s]

{'file': '/home/goksu/projects/ikincitik/results/run2/llm/rag_gemma-4-26b-a4b-qat-int4__simple.jsonl', 'records': 477, 'new': 2, 'skipped': 475, 'errors': 0, 'truncated': 0, 'seconds': 43}
----------------------------------------------------------------------------------------------------
q002 | gemma-4-26b-a4b-qat-int4 x simple | 18 tokens, 3.93 s, finish=stop
Q: Which subparts of the engine certification specifications apply to turbine engines?
A: The specifications of subparts A, D, E and F apply to Turbine Engines.
----------------------------------------------------------------------------------------------------
q001 | gemma-4-26b-a4b-qat-int4 x simple | 25 tokens, 4.583 s, finish=stop
Q: Besides the acceptable means of compliance in Book 2 of the engine certification specifications, which other document may also provide acceptable means of compliance to the Book 1 specifications?
A: AMC-20 may also provide acceptable means of compliance to the specifications in Book 1 of this CS

In [15]:
peek(rg.run_combination(used, "gemma", "robust", rg.combo_path(OUT, "gemma", "robust"), PARAMS, progress=tqdm))

gemma-4-26b-a4b-qat-int4 x robust:   0%|          | 0/2 [00:00<?, ?it/s]

{'file': '/home/goksu/projects/ikincitik/results/run2/llm/rag_gemma-4-26b-a4b-qat-int4__robust.jsonl', 'records': 477, 'new': 2, 'skipped': 475, 'errors': 0, 'truncated': 0, 'seconds': 57}
----------------------------------------------------------------------------------------------------
q002 | gemma-4-26b-a4b-qat-int4 x robust | 20 tokens, 2.357 s, finish=stop
Q: Which subparts of the engine certification specifications apply to turbine engines?
A: The specifications of subparts A, D, E and F apply to Turbine Engines [1].
----------------------------------------------------------------------------------------------------
q001 | gemma-4-26b-a4b-qat-int4 x robust | 27 tokens, 2.954 s, finish=stop
Q: Besides the acceptable means of compliance in Book 2 of the engine certification specifications, which other document may also provide acceptable means of compliance to the Book 1 specifications?
A: AMC-20 may also provide acceptable means of compliance to the specifications in Book 1 of th

## Combine the four files, check, summarise

**Checks** (each returns its problems; the cell stops if there are any, before it writes the main file):

1. every combination file has one record per question, no duplicates, no errors;
2. one model, one prompt and one prompt hash per file, and the hash is that of `rag_prompts.py` as it is now (a file made before a prompt edit fails here);
3. `retrieved` equals the X3 contexts' `hits` and `gold` the question file's, for every record;
4. no empty answer, and none cut off by `max_tokens`;
5. (after combining) `rag_output.jsonl` has the number of lines it should, each (question, LLM, prompt) once, and is the four files put together.

The summary shows, per combination, how many answers start with the abstention sentence of the robust prompt (`rag_prompts.NO_ANSWER`), for answerable and for unanswerable questions, and the mean tokens and seconds. It is for reading, not a check.

In [21]:
problems = {f"{k} x {p}": rg.check_combination(rg.combo_path(OUT, k, p), used, k, p) for k in LLM_ORDER for p in PROMPT_ORDER}
for name, probs in problems.items():
    print("PASS" if not probs else "FAIL", name, "" if not probs else f"({len(probs)}): {probs[:3]}")
assert not any(problems.values()), "fix the failing combinations above (run their cells again, or delete a stale file) and run this cell again"

n = rg.combine(OUT, used, LLM_ORDER, PROMPT_ORDER, OUT / "rag_output.jsonl")
final_problems = rg.check_combined(OUT / "rag_output.jsonl", OUT, used, LLM_ORDER, PROMPT_ORDER)
print("PASS" if not final_problems else "FAIL", "combined file:", n, "records", "" if not final_problems else final_problems)
assert not final_problems

summ = rg.summary(OUT, LLM_ORDER, PROMPT_ORDER)
summ.to_csv(OUT / "summary.csv", index=False, encoding="utf-8")
display(summ)
print(OUT / "rag_output.jsonl", "|", round((OUT / "rag_output.jsonl").stat().st_size / 1e6, 1), "MB")

PASS qwen x simple 
PASS qwen x robust 
PASS gemma x simple 
PASS gemma x robust 
PASS combined file: 1908 records 


,llm,prompt,records,answerable,unanswerable,abstained_answerable,abstained_unanswerable,truncated,errors,prompt_tokens,completion_tokens,seconds
0,qwen3.8-27b-int4,simple,477,469,8,0,0,0,0,1731.51,167.79,17.28
1,qwen3.8-27b-int4,robust,477,469,8,45,8,0,0,2330.51,89.85,15.05
2,gemma-4-26b-a4b-qat-int4,simple,477,469,8,0,0,0,0,1742.72,127.79,6.35
3,gemma-4-26b-a4b-qat-int4,robust,477,469,8,42,8,0,0,2344.72,135.74,6.71


/home/goksu/projects/ikincitik/results/run2/llm/rag_output.jsonl | 19.6 MB


In [22]:
out = rg.read_jsonl(OUT / "rag_output.jsonl")
unanswerable = next((r for r in out if not r["answerable"]), None)
for r in [out[0], *([unanswerable] if unanswerable else [])]:
    print("=" * 100)
    print(f"{r['question_id']} | {r['llm']} x {r['prompt']} | answerable: {r['answerable']} | finish={r['finish_reason']}")
    print("Q:", r["question"])
    print("A:", r["answer"])
    print("gold:", [(g["section_id"], g["role"]) for g in r["gold"]], "| distractors:", r["distractor_section_ids"])
    print("reference answer:", r["reference_answer"])
    print("retrieved:", [(h["rank"], h["unit_id"], round(h["score"], 3)) for h in r["retrieved"]])

q001 | qwen3.8-27b-int4 x simple | answerable: True | finish=stop
Q: Besides the acceptable means of compliance in Book 2 of the engine certification specifications, which other document may also provide acceptable means of compliance to the Book 1 specifications?
A: Based on document [1], **AMC-20** may also provide acceptable means of compliance to the specifications in Book 1 of the CS-E.
gold: [('AMC General (SUBPART A)', 'primary')] | distractors: []
reference answer: AMC-20.
retrieved: [(1, 'AMC General (SUBPART A)#w1', 0.591), (2, 'AMC E 80#w1', 0.056), (3, 'AMC E 790(a)(2)#w17', 0.029), (4, 'AMC E 910#w1', 0.027)]
q093 | qwen3.8-27b-int4 x simple | answerable: False | finish=stop
Q: What expansion space, as a percentage of capacity, is recommended for an engine-mounted hydraulic fluid reservoir?
A: Based on document [1], specifically section (2), the recommended expansion space is **10 percent** of the tank capacity.
gold: [] | distractors: ['AMC E 570', 'CS-E 570']
reference a

## Next

Stop vLLM when you are done (`docker compose stop vllm`). `results/<experiment>/llm/rag_output.jsonl` is the main file: one JSON object per line with `question_id`, `question`, `answer`, `gold`, `reference_answer`, `retrieved`, `llm`, `model_id`, `prompt`, `prompt_hash` and the token counts, seconds and parameters of the request. Read it with `rg.read_jsonl` or `pd.read_json(path, lines=True)`.